# The Claire Speech Engine: Engineering Validation
**Notebook:** COLAB-001
**Purpose:** Validate CSE installation, backend abstraction, and real speech generation using Fish Speech v1.5 and StyleTTS2.

> Run cells **in order**. Cell 1 requires a runtime restart before continuing.

## 1. Install Dependencies & Fish Speech v1.5
Clones Fish Speech v1.5.0, patches dependency pins, downloads model weights.

In [ ]:
!apt-get install -y -qq portaudio19-dev

import os
if not os.path.exists('fish-speech'):
    !git clone https://github.com/fishaudio/fish-speech.git
    !cd fish-speech && git checkout v1.5.0
else:
    print('fish-speech/ already cloned, skipping.')

!pip install -q -U huggingface_hub
!hf download fishaudio/fish-speech-1.5 --local-dir checkpoints/fish-speech-1.5

# Verify checkpoint
expected = 'checkpoints/fish-speech-1.5/firefly-gan-vq-fsq-8x1024-21hz-generator.pth'
assert os.path.exists(expected), f'Checkpoint missing: {expected}'
print(f'\u2705 Checkpoint verified: {expected}')

# Patch Fish Speech dependency pins
import re
for fp in ['fish-speech/pyproject.toml', 'fish-speech/requirements.txt']:
    if os.path.exists(fp):
        with open(fp, 'r') as f:
            content = f.read()
        for pkg in ['tokenizers', 'transformers', 'torch', 'numpy']:
            pattern = pkg + r'(?:\[[^\]]*\])?(?:\s*[><=!~]+\s*[\w.*]+(?:\s*,\s*[><=!~]+\s*[\w.*]+)*)'
            content = re.sub(pattern, pkg, content)
        with open(fp, 'w') as f:
            f.write(content)
        print(f'Patched: {fp}')

!pip install -q --no-build-isolation -e fish-speech
!pip install -q styletts2 soundfile

# Fix numpy/scipy
!rm -rf /usr/local/lib/python3.12/dist-packages/numpy*
!rm -rf /usr/local/lib/python3.12/dist-packages/scipy*
!pip install -q 'numpy<2.1' scipy

print('\n' + '='*60)
print('\u2705 All dependencies installed.')
print('\u26a0\ufe0f  NOW: Runtime \u2192 Restart session, then skip Cell 1 and run Cell 2.')
print('='*60)

## 2. Mount Google Drive & Verify Voice Assets
Mount your Drive to access Claire's reference audio files.

In [ ]:
import sys, os
if '/content/fish-speech' not in sys.path:
    sys.path.insert(0, '/content/fish-speech')

from google.colab import drive
drive.mount('/content/drive')

VOICES_DIR = '/content/drive/MyDrive/claire/voices'
os.makedirs(VOICES_DIR, exist_ok=True)

voice_files = [f for f in os.listdir(VOICES_DIR) if f.endswith('.wav')] if os.path.exists(VOICES_DIR) else []
print(f'\n\U0001f3a4 Voice references found: {len(voice_files)}')
for vf in sorted(voice_files):
    print(f'  - {vf}')

if not voice_files:
    print(f'\n\u26a0\ufe0f  No voice references found! Upload claire_neutral.wav to: {VOICES_DIR}')

import torch
print(f'\n\u2705 PyTorch {torch.__version__}')
print(f'\u2705 CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'\u2705 GPU: {torch.cuda.get_device_name(0)}')
    print(f'\u2705 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\n\u2705 Google Drive mounted and ready.')

## 3. Install CSE & Register Voice

In [ ]:
!pip install -q git+https://github.com/Yukariii04/Claire-Speech-Engine.git

from cse.voice import register_voice_package, VoicePackage, VoiceMetadata
from pathlib import Path

meta = VoiceMetadata(
    id='default', name='Default', version='1.0.0',
    author='CSE', language='en', backend='fishspeech',
    sample_rate=24000, channels=1,
    description='Evaluation voice', license='MIT'
)
try:
    register_voice_package(VoicePackage(metadata=meta, path=Path('.')))
except Exception:
    pass

print('\u2705 CSE installed and voice registered.')

## 4. Fish Speech Validation
Generate real speech using Fish Speech v1.5 through the CSE public API.

In [ ]:
import os
os.environ['FISH_SPEECH_DIR'] = '/content/fish-speech'
os.environ['FISH_CHECKPOINT_DIR'] = '/content/checkpoints/fish-speech-1.5'
os.environ['VOICES_DIR'] = '/content/drive/MyDrive/claire/voices'

from cse import SpeechEngine
from IPython.display import Audio, display
from google.colab import files

engine = SpeechEngine()
engine.load_backend('fishspeech')
engine.load_voice('default')

print('Generating speech via Fish Speech v1.5...')
speech_fs = engine.speak('Hello from the Claire Speech Engine. This is a validation test using Fish Speech.')

print(f'Success: {speech_fs.success}')
print(f'Audio: {speech_fs.audio_path}')
print(f'Duration: {speech_fs.duration_seconds:.2f}s')

if speech_fs.audio_path and os.path.exists(speech_fs.audio_path):
    display(Audio(str(speech_fs.audio_path)))
    # Save a copy for download
    import shutil
    shutil.copy(speech_fs.audio_path, 'validation_fishspeech.wav')
    print('\u2705 Fish Speech validation complete.')

## 5. StyleTTS2 Validation
Switch to StyleTTS2 and generate speech — no reference audio needed.

In [ ]:
engine.load_backend('styletts2')
engine.load_voice('default')

print('Generating speech via StyleTTS2...')
speech_st = engine.speak('Hello from the Claire Speech Engine. This is a validation test using Style TTS 2.')

print(f'Success: {speech_st.success}')
print(f'Audio: {speech_st.audio_path}')
print(f'Duration: {speech_st.duration_seconds:.2f}s')

if speech_st.audio_path and os.path.exists(speech_st.audio_path):
    display(Audio(str(speech_st.audio_path)))
    import shutil
    shutil.copy(speech_st.audio_path, 'validation_styletts2.wav')
    print('\u2705 StyleTTS2 validation complete.')

## 6. Download Outputs

In [ ]:
from google.colab import files
import os

for f in ['validation_fishspeech.wav', 'validation_styletts2.wav']:
    if os.path.exists(f):
        print(f'Downloading {f}...')
        files.download(f)

engine.shutdown()
print('\u2705 Done. Both validation WAVs downloaded.')